# GreenAudioBench — official Colab T4 runner

Thin stage runner. **Runtime → Change runtime type → T4 GPU** first.

Run cells top to bottom. Set `STAGE` in the config cell:

| STAGE | What runs | Needs |
|---|---|---|
| `smoke` | tiny CPU/GPU smoke of every stage, writes only to `data/smoke/` | — |
| `m2` | ESC-50 embedding extraction (all 5 models, fp32) | ESC-50 (auto-downloaded) |
| `m3` | linear probes on cached embeddings | `m2` caches |
| `m4` | zero-shot CLAP (both models, 2 templates) | ESC-50 (+`m2` cache speeds it up) |
| `m5` | latency + NVML energy (fp32+fp16, batch 1/32) | ESC-50 |

Stages are idempotent; caches and results survive re-runs. With
`USE_DRIVE=False` embedding caches + results also persist across Colab VMs.

In [ ]:
# ------------------------- CONFIG -------------------------

REPO_URL = "https://github.com/IzbassarO/greenaudiobench.git"
EXPECTED_HEAD = "82d940fba85070478e49a050e4d70d8b8138940c"

STAGE = "smoke"           # smoke | m2 | m3 | m4 | m5

USE_DRIVE = False         # KEEP FALSE — everything stays in this temporary VM
ALLOW_DIRTY = False       # never enable for official runs

assert REPO_URL

In [ ]:
# ------------------- GPU / driver check -------------------
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No CUDA device — select the T4 GPU runtime"
name = torch.cuda.get_device_name(0)
print("GPU:", name, "| torch", torch.__version__, "| CUDA", torch.version.cuda)
if "T4" not in name:
    print("WARNING: official numbers are defined on Tesla T4 — this GPU is", name)

In [ ]:
# ---------------- clone + pin + dependencies ----------------

import os
import subprocess

if not os.path.isdir("/content/greenaudiobench"):
    !git clone {REPO_URL} /content/greenaudiobench

%cd /content/greenaudiobench

# Run exactly the independently reviewed commit.
!git fetch origin
!git checkout --detach {EXPECTED_HEAD}

head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()

print("HEAD:", head)
assert head == EXPECTED_HEAD, f"Wrong commit: {head}"

!pip install -q -r env/requirements.txt

# Keep this OUTSIDE the repository so provenance remains clean.
!pip freeze > /content/requirements-lock-colab.txt

dirty = subprocess.run(
    [
        "git", "status", "--porcelain", "--", ".",
        ":(exclude)results",
        ":(exclude)figures",
    ],
    capture_output=True,
    text=True,
).stdout.strip()

print("working tree:", "DIRTY" if dirty else "clean")
assert not dirty, f"Repository became dirty:\n{dirty}"

print("✓ Reviewed GreenAudioBench revision ready")

In [ ]:
# ------- optional: persist caches/results to Drive ---------
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    persist = "/content/drive/MyDrive/greenaudiobench"
    for sub in ("data/embeddings", "results", "figures"):
        src = os.path.join(persist, sub)
        os.makedirs(src, exist_ok=True)
        if os.path.isdir(sub) and not os.path.islink(sub):
            !cp -rn {sub}/. {src}/ 2>/dev/null || true
            !rm -rf {sub}
        if not os.path.islink(sub):
            os.makedirs(os.path.dirname(sub), exist_ok=True)
            os.symlink(src, sub)
    print("persisted dirs linked to", persist)

In [ ]:
# ----------------- data (ESC-50 only, M1) ------------------
if STAGE in ("smoke", "m2", "m4", "m5"):
    !python scripts/download_data.py --datasets esc50

In [ ]:
# ---------------------- run the stage ----------------------
dirty_flag = "--allow-dirty" if ALLOW_DIRTY else ""
cmds = {
    "smoke": [
        "python -m pytest -q",
        "python scripts/extract_embeddings.py --dataset esc50 --smoke 8",
        "python scripts/run_probes.py --dataset esc50 --smoke",
        f"python scripts/run_zeroshot.py --dataset esc50 --smoke 8",
        f"python scripts/measure_efficiency.py --smoke",
    ],
    "m2": [f"python scripts/extract_embeddings.py --dataset esc50 --batch-size 16 {dirty_flag}"],
    "m3": [f"python scripts/run_probes.py --dataset esc50 {dirty_flag}"],
    "m4": [f"python scripts/run_zeroshot.py --dataset esc50 {dirty_flag}"],
    "m5": [f"python scripts/measure_efficiency.py {dirty_flag}"],
}
for cmd in cmds[STAGE]:
    print("\n>>>", cmd, flush=True)
    rc = os.system(cmd)
    assert rc == 0, f"stage command failed: {cmd}"

In [ ]:
# ---------------- collect reproducibility artifacts ----------------

import os
import shutil
import subprocess

ART = "/content/greenaudiobench_artifacts"

# Prevent stale/nested copies from previous packaging runs.
shutil.rmtree(ART, ignore_errors=True)
os.makedirs(ART, exist_ok=True)

# Results
if os.path.isdir("results"):
    shutil.copytree("results", f"{ART}/results")

# Figures
if os.path.isdir("figures"):
    shutil.copytree("figures", f"{ART}/figures")

# Expensive M2 outputs — preserve them.
if os.path.isdir("data/embeddings"):
    shutil.copytree("data/embeddings", f"{ART}/embeddings")

# Reproducibility metadata
for src in [
    "data/CHECKSUMS.txt",
    "env/MODELS.md",
]:
    if os.path.exists(src):
        shutil.copy2(src, ART)

if os.path.exists("/content/requirements-lock-colab.txt"):
    shutil.copy2(
        "/content/requirements-lock-colab.txt",
        f"{ART}/requirements-lock-colab.txt",
    )

# Exact code revision
head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True
).strip()

with open(f"{ART}/GIT_HEAD.txt", "w") as f:
    f.write(head + "\n")

# GPU information
with open(f"{ART}/NVIDIA_SMI.txt", "w") as f:
    subprocess.run(["nvidia-smi"], stdout=f, text=True)

# Package everything.
!cd /content && rm -f greenaudiobench_artifacts.zip
!cd /content && zip -qr greenaudiobench_artifacts.zip greenaudiobench_artifacts

!du -sh /content/greenaudiobench_artifacts
!ls -lh /content/greenaudiobench_artifacts.zip

print()
print("READY:")
print("/content/greenaudiobench_artifacts.zip")
print("Download it from the Colab Files panel.")